## BERT

### Masked Language Model

In [1]:
data = [
    "i am a student",
    "how are you",
    "i love machine learning",
    "good morning",
    "thank you",
    "see you later",
    "what is your name",
    "where are you going",
    "i like coffee",
    "welcome"
]

In [2]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Dense,
    LayerNormalization,
    MultiHeadAttention
)

from tensorflow.keras import Model

In [3]:
vocab_size = 1000
sequence_length = 20

vectorizer = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length
)

vectorizer.adapt(data)

tokenized_sentences = vectorizer(data)

In [4]:
mask_token_id = 1

inputs = tokenized_sentences.numpy().copy()
targets = tokenized_sentences.numpy().copy()

for sentence in inputs:
    pos = np.random.randint(1,5)
    sentence[pos] = mask_token_id

inputs = tf.constant(inputs)
targets = tf.constant(targets)

In [5]:
class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(
        self,
        sequence_length,
        vocab_size,
        embed_dim
    ):
        super().__init__()

        self.token_embedding = Embedding(
            vocab_size,
            embed_dim
        )

        self.position_embedding = Embedding(
            sequence_length,
            embed_dim
        )

    def call(self, inputs):

        length = tf.shape(inputs)[-1]

        positions = tf.range(
            start=0,
            limit=length,
            delta=1
        )

        embedded_tokens = self.token_embedding(
            inputs
        )

        embedded_positions = self.position_embedding(
            positions
        )

        return embedded_tokens + embedded_positions

In [6]:
# Encoder block

class BERTEncoder(tf.keras.layers.Layer):

    def __init__(
        self,
        embed_dim,
        dense_dim,
        num_heads
    ):
        super().__init__()

        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(
                dense_dim,
                activation="relu"
            ),
            Dense(embed_dim)
        ])

        self.layernorm1 = LayerNormalization()
        self.layernorm2 = LayerNormalization()

    def call(self, inputs):

        attention_output = self.attention(
            inputs,
            inputs
        )

        out1 = self.layernorm1(
            inputs + attention_output
        )

        ffn_output = self.ffn(out1)

        return self.layernorm2(
            out1 + ffn_output
        )

In [7]:
embed_dim = 128
dense_dim = 256
num_heads = 4

bert_input = tf.keras.Input(
    shape=(None,),
    dtype="int64"
)

x = PositionalEmbedding(
    sequence_length,
    vocab_size,
    embed_dim
)(bert_input)

x = BERTEncoder(
    embed_dim,
    dense_dim,
    num_heads
)(x)

output = Dense(
    vocab_size,
    activation="softmax"
)(x)

bert = Model(
    bert_input,
    output
)

In [8]:
bert.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

bert.fit(
    inputs,
    targets,
    batch_size=2,
    epochs=30
)

Epoch 1/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.6600 - loss: 4.6624   
Epoch 2/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8500 - loss: 2.3062 
Epoch 3/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8500 - loss: 1.5979 
Epoch 4/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8500 - loss: 1.2249 
Epoch 5/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8500 - loss: 1.0423
Epoch 6/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8500 - loss: 0.9329 
Epoch 7/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8500 - loss: 0.8127 
Epoch 8/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8500 - loss: 0.6736 
Epoch 9/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8850 - loss: 0.5576 
Epoch 10/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8850 - loss: 0.4620 
Epoch 11/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9250 - loss: 0.3819 
Epoch 12/30
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9350 - loss: 0.3164 

In [9]:
sentence = ["i mask coffee"]

tokenized = vectorizer(sentence)

prediction = bert.predict(tokenized)

predicted_ids = np.argmax(
    prediction,
    axis=-1
)

print(predicted_ids)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
[[ 3  0 23  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]]
